# 02 — Pooled LSTM for PM Forecasting

Train one PyTorch LSTM across stations to jointly forecast PM10 and PM2.5. Windows are built within chronological splits, preprocessing is fitted on training data only, and all test metrics are reported in the original concentration scale.

## 1. Configuration

In [1]:
from pathlib import Path

# Main experiment settings; change HORIZON to 6 or 24 to run another forecast horizon.
LOOKBACK = 48
HORIZON = 1
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
HIDDEN = 64
LAYERS = 2
DROPOUT = 0.2
STATION_SUBSET = None  # None uses every station; e.g. ["E421"] for a quick single-station debug run.
RANDOM_SEED = 42

DATA_PATH = Path(r"C:\Users\jains\Downloads\Air quality dataset\cleaned\air_quality_all_stations_cleaned.csv")
RESULTS_DIR = Path.cwd() / "lstm_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA_PATH}")
print(f"Results: {RESULTS_DIR.resolve()}")

Data: C:\Users\jains\Downloads\Air quality dataset\cleaned\air_quality_all_stations_cleaned.csv
Results: E:\coding\DL2project\lstm_results


## 2. Imports, seeds, and device

In [2]:
import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(8, torch.get_num_threads()))
print(f"PyTorch: {torch.__version__}; device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'torch'

## 3. Load data and make shared chronological splits

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Cleaned data file not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
if STATION_SUBSET is not None:
    missing_stations = set(STATION_SUBSET) - set(df["station"].unique())
    if missing_stations:
        raise ValueError(f"Unknown station(s): {sorted(missing_stations)}")
    df = df[df["station"].isin(STATION_SUBSET)].copy()

FEATURES = [
    "PM10", "PM2.5", "temperature", "rain", "pressure", "precipitation",
    "wind_speed", "clouds_ordinal", "hour_sin", "hour_cos",
    "day_of_year_sin", "day_of_year_cos",
]
TARGETS = ["PM10", "PM2.5"]
required_columns = ["datetime", "station", *FEATURES]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from cleaned data: {missing_columns}")
# Directional wind fields are intentionally excluded; wind_speed remains a feature.
df = df[required_columns].sort_values(["station", "datetime"]).reset_index(drop=True)
if df.duplicated(["station", "datetime"]).any():
    raise ValueError("Duplicate station/timestamp keys must be resolved before window creation.")

all_timestamps = pd.DatetimeIndex(df["datetime"].drop_duplicates().sort_values())
train_cut_idx = int(len(all_timestamps) * 0.70)
validation_cut_idx = int(len(all_timestamps) * 0.85)
if train_cut_idx <= 0 or validation_cut_idx <= train_cut_idx or validation_cut_idx >= len(all_timestamps):
    raise ValueError("Not enough distinct timestamps to create 70/15/15 chronological splits.")
TRAIN_START, TRAIN_END = all_timestamps[0], all_timestamps[train_cut_idx - 1]
VAL_START, VAL_END = all_timestamps[train_cut_idx], all_timestamps[validation_cut_idx - 1]
TEST_START, TEST_END = all_timestamps[validation_cut_idx], all_timestamps[-1]
print(f"Rows: {len(df):,}; stations: {df['station'].nunique()}; features: {len(FEATURES)}")
print(f"Train:      {TRAIN_START} through {TRAIN_END}")
print(f"Validation: {VAL_START} through {VAL_END}")
print(f"Test:       {TEST_START} through {TEST_END}")
print("Every station uses these same chronological cut dates.")
print(f"Wind-direction input columns retained: {[c for c in df.columns if 'wind_direction' in c]}")

## 4. Train-only scaling and split-local window datasets

In [ ]:
class WindowDataset(Dataset):
    """Lazily slice windows so the full overlapping window tensor stays out of memory."""
    def __init__(self, station_arrays, records, horizon):
        self.station_arrays = station_arrays
        self.records = records
        self.horizon = horizon

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        station_idx, origin_idx, target_idx = self.records[index]
        arrays = self.station_arrays[station_idx]
        x_start = origin_idx - LOOKBACK + 1
        x = arrays["x"][x_start:origin_idx + 1]
        y_scaled = arrays["y_scaled"][target_idx]
        y_original = arrays["y_original"][target_idx]
        persistence = arrays["persistence"][origin_idx]
        return (
            torch.from_numpy(x), torch.from_numpy(y_scaled),
            torch.tensor(station_idx, dtype=torch.long),
            torch.from_numpy(y_original), torch.from_numpy(persistence),
        )

def build_split_datasets(horizon):
    """Fit station scalers on train only and create windows contained in each split."""
    if horizon < 1:
        raise ValueError("HORIZON must be >= 1")
    split_bounds = {
        "train": (TRAIN_START, TRAIN_END),
        "validation": (VAL_START, VAL_END),
        "test": (TEST_START, TEST_END),
    }
    stations = sorted(df["station"].unique())
    station_arrays = {name: [] for name in split_bounds}
    records = {name: [] for name in split_bounds}
    station_scalers = {}
    for station_idx, station in enumerate(stations):
        station_df = df.loc[df["station"].eq(station)].sort_values("datetime").reset_index(drop=True)
        train_mask = station_df["datetime"].between(TRAIN_START, TRAIN_END)
        if train_mask.sum() <= LOOKBACK + horizon:
            raise ValueError(f"Station {station} has too few training rows for the configured window and horizon.")
        train_features = station_df.loc[train_mask, FEATURES].astype(float)
        medians = train_features.median()
        if medians.isna().any():
            bad = medians.index[medians.isna()].tolist()
            raise ValueError(f"Training features entirely missing for station {station}: {bad}")
        feature_scaler = StandardScaler()
        feature_scaler.fit(train_features.fillna(medians).to_numpy(dtype=np.float64))
        target_scalers = {}
        for target in TARGETS:
            valid_train_target = station_df.loc[train_mask, target].dropna().to_numpy(dtype=np.float64).reshape(-1, 1)
            if len(valid_train_target) == 0:
                raise ValueError(f"No valid training targets for {station} / {target}")
            scaler = StandardScaler().fit(valid_train_target)
            target_scalers[target] = scaler
        station_scalers[station] = {"features": feature_scaler, "targets": target_scalers, "medians": medians}

        # Transform each split independently; no window can cross a split boundary.
        for split_name, (start, end) in split_bounds.items():
            part = station_df.loc[station_df["datetime"].between(start, end)].reset_index(drop=True)
            raw_x = part[FEATURES].astype(float).fillna(medians).to_numpy(dtype=np.float64)
            x_scaled = feature_scaler.transform(raw_x).astype(np.float32)
            y_original = part[TARGETS].to_numpy(dtype=np.float64)
            y_scaled = np.full(y_original.shape, np.nan, dtype=np.float32)
            persistence = part[TARGETS].to_numpy(dtype=np.float64)
            for target_idx, target in enumerate(TARGETS):
                valid = np.isfinite(y_original[:, target_idx])
                target_scaler = target_scalers[target]
                y_scaled[valid, target_idx] = target_scaler.transform(
                    y_original[valid, target_idx].reshape(-1, 1)
                ).ravel().astype(np.float32)
                # The last input observation is the persistence forecast origin.
                input_column_idx = FEATURES.index(target)
            # Persistence is the observed target at the forecast origin; keep missing origins NaN.
            persistence[:, target_idx] = part[target].to_numpy(dtype=np.float64)
            local_arrays = {"x": x_scaled, "y_scaled": y_scaled, "y_original": y_original.astype(np.float32), "persistence": persistence.astype(np.float32)}
            station_arrays[split_name].append(local_arrays)
            final_target = len(part) - 1
            first_target = LOOKBACK - 1 + horizon
            for target_pos in range(first_target, len(part)):
                origin_pos = target_pos - horizon
                if np.isfinite(y_original[target_pos]).all():
                    records[split_name].append((station_idx, origin_pos, target_pos))

    datasets = {
        name: WindowDataset(station_arrays[name], records[name], horizon)
        for name in split_bounds
    }
    return datasets, station_scalers, stations

print(f"Window builder ready: lookback={LOOKBACK}, configured horizon={HORIZON}.")

## 5. Model, training, evaluation, plots, and saving

In [ ]:
class PooledLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN, layers=LAYERS, dropout=DROPOUT, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        return self.head(sequence_output[:, -1, :])

def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    if not len(actual):
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n": 0}
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted) if len(actual) >= 2 else np.nan,
        "n": len(actual),
    }

def run_experiment(horizon):
    """Train one pooled LSTM at a horizon and save predictions, metrics, and best weights."""
    set_seed(RANDOM_SEED)
    started = time.time()
    datasets, station_scalers, stations = build_split_datasets(horizon)
    if any(len(datasets[name]) == 0 for name in ("train", "validation", "test")):
        raise ValueError(f"At least one split has no valid windows for horizon {horizon}.")
    print(f"Horizon {horizon}: windows train/validation/test = "
          f"{len(datasets['train']):,}/{len(datasets['validation']):,}/{len(datasets['test']):,}")
    train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                              pin_memory=DEVICE.type == "cuda", generator=torch.Generator().manual_seed(RANDOM_SEED))
    val_loader = DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                            pin_memory=DEVICE.type == "cuda")
    test_loader = DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                             pin_memory=DEVICE.type == "cuda")

    model = PooledLSTM(len(FEATURES)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history = {"train_loss": [], "validation_loss": []}
    weights_path = RESULTS_DIR / f"lstm_horizon_{horizon}_best.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for x, y, _, _, _ in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(x)
            loss = criterion(prediction, y)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * len(x)
            train_count += len(x)
        train_loss = train_loss_sum / train_count

        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        with torch.no_grad():
            for x, y, _, _, _ in val_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                val_loss_sum += criterion(model(x), y).item() * len(x)
                val_count += len(x)
        val_loss = val_loss_sum / val_count
        scheduler.step(val_loss)
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, weights_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        print(f"H={horizon:>2} | epoch {epoch:02d}/{EPOCHS} | train={train_loss:.5f} | "
              f"val={val_loss:.5f} | lr={optimizer.param_groups[0]['lr']:.2e}")
        if epochs_without_improvement >= 8:
            print("Early stopping: validation loss did not improve for 8 epochs.")
            break

    if best_state is None:
        raise RuntimeError("Training ended without a finite validation checkpoint.")
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
    model.eval()
    prediction_scaled, actual_original, persistence_original, station_indices = [], [], [], []
    with torch.no_grad():
        for x, y_scaled, sid, y_original, persistence in test_loader:
            prediction_scaled.append(model(x.to(DEVICE, non_blocking=True)).cpu().numpy())
            actual_original.append(y_original.numpy())
            persistence_original.append(persistence.numpy())
            station_indices.append(sid.numpy())
    prediction_scaled = np.concatenate(prediction_scaled)
    actual_original = np.concatenate(actual_original)
    persistence_original = np.concatenate(persistence_original)
    station_indices = np.concatenate(station_indices)
    station_names = np.asarray(stations, dtype=object)[station_indices]

    predictions_original = np.empty_like(prediction_scaled, dtype=np.float64)
    for station_idx, station in enumerate(stations):
        station_rows = station_indices == station_idx
        if not station_rows.any():
            continue
        for target_idx, target in enumerate(TARGETS):
            scaler = station_scalers[station]["targets"][target]
            predictions_original[station_rows, target_idx] = scaler.inverse_transform(
                prediction_scaled[station_rows, target_idx].reshape(-1, 1)
            ).ravel()

    metrics = []
    for target_idx, target in enumerate(TARGETS):
        for group_name, row_mask in [("overall", np.ones(len(station_names), dtype=bool))] + [
            (station, station_names == station) for station in stations
        ]:
            for method, estimate in [("LSTM", predictions_original[:, target_idx]),
                                     ("Persistence", persistence_original[:, target_idx])]:
                values = regression_metrics(actual_original[row_mask, target_idx], estimate[row_mask])
                metrics.append({"horizon": horizon, "target": target, "station": group_name,
                                "method": method, **values})
    metrics_df = pd.DataFrame(metrics)
    metrics_path = RESULTS_DIR / f"lstm_horizon_{horizon}_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False)

    loss_path = RESULTS_DIR / f"lstm_horizon_{horizon}_loss.png"
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["train_loss"], label="Train")
    ax.plot(history["validation_loss"], label="Validation")
    ax.set(title=f"Training history — horizon {horizon}", xlabel="Epoch", ylabel="MSE (scaled targets)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(loss_path, dpi=150)
    plt.show()

    prediction_frame = pd.DataFrame({
        "station": station_names,
        "PM10_actual": actual_original[:, 0], "PM10_predicted": predictions_original[:, 0],
        "PM2.5_actual": actual_original[:, 1], "PM2.5_predicted": predictions_original[:, 1],
        "PM10_persistence": persistence_original[:, 0], "PM2.5_persistence": persistence_original[:, 1],
    })
    test_dates = []
    # Dataset order is station-major, then chronological within each station.
    for station in stations:
        part = df.loc[df["station"].eq(station) & df["datetime"].between(TEST_START, TEST_END)].reset_index(drop=True)
        first_target = LOOKBACK - 1 + horizon
        valid_positions = [pos for pos in range(first_target, len(part)) if np.isfinite(part.loc[pos, TARGETS].to_numpy(dtype=float)).all()]
        test_dates.extend(part.loc[valid_positions, "datetime"].tolist())
    prediction_frame["datetime"] = test_dates
    prediction_path = RESULTS_DIR / f"lstm_horizon_{horizon}_test_predictions.csv"
    prediction_frame.to_csv(prediction_path, index=False)

    plot_stations = stations[:2]
    plot_start = TEST_START
    plot_end = min(TEST_END, plot_start + pd.Timedelta(days=7))
    fig, axes = plt.subplots(len(plot_stations), 2, figsize=(15, 4 * len(plot_stations)), squeeze=False)
    for row, station in enumerate(plot_stations):
        plotted = prediction_frame.loc[
            prediction_frame["station"].eq(station)
            & prediction_frame["datetime"].between(plot_start, plot_end)
        ].sort_values("datetime")
        for col, target in enumerate(TARGETS):
            axes[row, col].plot(plotted["datetime"], plotted[f"{target}_actual"], label="Actual", linewidth=1)
            axes[row, col].plot(plotted["datetime"], plotted[f"{target}_predicted"], label="LSTM", linewidth=1)
            axes[row, col].set(title=f"{station} — {target}", xlabel="Test date", ylabel="Concentration")
            axes[row, col].legend()
    fig.suptitle(f"One-week test predictions — horizon {horizon}")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"lstm_horizon_{horizon}_one_week.png", dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for target_idx, target in enumerate(TARGETS):
        axes[target_idx].scatter(actual_original[:, target_idx], predictions_original[:, target_idx], alpha=0.15, s=8)
        low = min(actual_original[:, target_idx].min(), predictions_original[:, target_idx].min())
        high = max(actual_original[:, target_idx].max(), predictions_original[:, target_idx].max())
        axes[target_idx].plot([low, high], [low, high], "r--", linewidth=1)
        axes[target_idx].set(title=f"{target} — horizon {horizon}", xlabel="Actual", ylabel="Predicted")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"lstm_horizon_{horizon}_scatter.png", dpi=150)
    plt.show()

    print(f"\nHorizon {horizon} test metrics (original scale):")
    display(metrics_df.loc[metrics_df["station"].eq("overall")].pivot(index="target", columns="method", values=["MAE", "RMSE", "R2"]))
    print(f"Saved weights: {weights_path}")
    print(f"Saved metrics: {metrics_path}")
    print(f"Elapsed: {(time.time() - started) / 60:.1f} minutes")
    return {"horizon": horizon, "metrics": metrics_df, "history": history,
            "predictions": prediction_frame, "weights_path": weights_path}

print("LSTM and experiment runner are ready.")

## 6. Train and evaluate — horizon 1

In [ ]:
results_by_horizon = {}
results_by_horizon[HORIZON] = run_experiment(HORIZON)

## 7. Train and evaluate — horizon 6

In [ ]:
results_by_horizon[6] = run_experiment(6)

## 8. Train and evaluate — horizon 24

In [ ]:
results_by_horizon[24] = run_experiment(24)

## 9. Final horizon comparison

In [ ]:
all_metrics = pd.concat([results_by_horizon[h]["metrics"] for h in [1, 6, 24]], ignore_index=True)
comparison = all_metrics.loc[all_metrics["station"].eq("overall")].sort_values(["target", "method", "horizon"])
display(comparison.reset_index(drop=True))
comparison_path = RESULTS_DIR / "lstm_horizon_comparison.csv"
all_metrics.to_csv(RESULTS_DIR / "lstm_all_horizons_metrics.csv", index=False)
comparison.to_csv(comparison_path, index=False)
print(f"Saved final horizon comparison to {comparison_path}")
print("All model inputs and targets were transformed using station-specific scalers fitted only on the training period; reported metrics are in original units.")

### Run results (CPU, seed 42)

The configured split dates were train **2024-05-02 20:00–2025-06-18 01:00**, validation **2025-06-18 02:00–2025-09-14 04:00**, and test **2025-09-14 05:00–2025-12-11 07:00**. Best validation checkpoints were at epochs 16 (H=1), 7 (H=6), and 2 (H=24); early stopping occurred after epochs 24, 15, and 10, respectively.

| Target | Horizon (h) | Method | MAE | RMSE | R² |
|---|---:|---|---:|---:|---:|
| PM10 | 1 | LSTM | 2.758 | 4.892 | 0.837 |
| PM10 | 1 | Persistence | 2.884 | 5.398 | 0.801 |
| PM10 | 6 | LSTM | 6.154 | 9.003 | 0.447 |
| PM10 | 6 | Persistence | 7.313 | 11.435 | 0.105 |
| PM10 | 24 | LSTM | 7.676 | 10.511 | 0.249 |
| PM10 | 24 | Persistence | 7.257 | 10.865 | 0.191 |
| PM2.5 | 1 | LSTM | 1.975 | 3.676 | 0.882 |
| PM2.5 | 1 | Persistence | 2.056 | 4.037 | 0.857 |
| PM2.5 | 6 | LSTM | 4.580 | 7.180 | 0.550 |
| PM2.5 | 6 | Persistence | 5.627 | 9.369 | 0.230 |
| PM2.5 | 24 | LSTM | 5.728 | 8.440 | 0.380 |
| PM2.5 | 24 | Persistence | 5.545 | 8.706 | 0.334 |

Metrics above are pooled across test stations and use original concentration units. Per-station metrics are in `lstm_all_horizons_metrics.csv`; weights, predictions, and plots are saved with the horizon in their filenames under `lstm_results/`.